# Full architecture demo

This notebook shows the complete workflow we built:
- standardized data loading
- one `Universe` object with shared prices and returns
- multiple constructions stored on the same universe
- per-universe output folders
- fixed-weight backtesting
- Monte Carlo simulation
- evaluation results attached back into each stored construction

In [1]:
from pathlib import Path

from portafolios import Universe
from portafolios.data.local_loader import local_loader
from portafolios.constructores import EqualWeightConstructor, Markowitz, NaiveRiskParity
from portafolios.constructores.hrp_style.hrp_core import HRPStyle
from portafolios.eval import Backtester, MonteCarloEngine

PROJECT_ROOT = Path.cwd()
CSV_PATH = PROJECT_ROOT / "data_clean_stock_data.csv"

print("Project root:", PROJECT_ROOT)
print("CSV exists:", CSV_PATH.exists())

Project root: c:\Users\narro\Desktop\semestre\honores_actuaria
CSV exists: True


## 1. Create one universe

This is the same edited main object you were already using, now with:
- shared market data
- stored constructions
- evaluation hooks
- its own output folder tree

In [2]:
u = Universe(
    universe_name="thesis_full_demo",
    base_output_dir=PROJECT_ROOT / "results",
    tickers=["AAPL", "MSFT", "AMZN", "GOOG"],
    start="2020-01-01",
    end="2020-12-31",
    loader=local_loader,
    loader_kwargs={"path": CSV_PATH, "prefer_adj_close": True},
    freq="B",
).preparar_datos()

u.prices.head()

,AAPL,MSFT,AMZN,GOOG
Date,,,,
2020-01-02,72.776606,153.428246,94.900497,68.186813
2020-01-03,72.771729,152.683705,94.309998,68.439404
2020-01-06,72.621654,151.872308,95.184502,69.663459
2020-01-07,72.849231,152.416407,95.694504,69.921535
2020-01-08,73.706271,153.495104,95.550003,70.337515


In [3]:
print("Universe name:", u.universe_name)
print("Output dir:", u.output_dir)
print("Data dir:", u.data_dir)
print("Constructions dir:", u.constructions_dir)
print("Backtests dir:", u.backtests_dir)
print("Monte Carlo dir:", u.monte_carlo_dir)
print("Plots dir:", u.plots_dir)
print("Returns shape:", u.returns.shape)

Universe name: thesis_full_demo
Output dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo
Data dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\data
Constructions dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\constructions
Backtests dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\backtests
Monte Carlo dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\monte_carlo
Plots dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\plots
Returns shape: (260, 4)


## 2. Build multiple constructions on the same universe

Each construction stores:
- weights
- selected assets
- parameters
- in-sample metrics
- construction window
- placeholders for backtest and Monte Carlo results

In [4]:
construction_kwargs = {
    "ann_factor": 252,
    "construction_start": "2020-01-01",
    "construction_end": "2020-06-30",
}

u.construir(EqualWeightConstructor(), label="equal_weight", **construction_kwargs)
u.construir(Markowitz(), label="markowitz", ret_kind="simple", allow_short=False, **construction_kwargs)
u.construir(NaiveRiskParity(), label="naive_risk_parity", **construction_kwargs)
u.construir(HRPStyle(), label="hrp_style", **construction_kwargs)

u.list_constructions()

['equal_weight', 'markowitz', 'naive_risk_parity', 'hrp_style']

In [5]:
for name in u.list_constructions():
    result = u.get_construction(name)
    print(name, "|", result.method, "| window:", result.construction_start.date(), "->", result.construction_end.date())

equal_weight | equal_weight | window: 2020-01-01 -> 2020-06-30
markowitz | markowitz_max_sharpe | window: 2020-01-01 -> 2020-06-30
naive_risk_parity | Naive Risk Parity (1/sigma) | window: 2020-01-01 -> 2020-06-30
hrp_style | HRP-style | window: 2020-01-01 -> 2020-06-30


## 3. Inspect one stored construction

In [6]:
markowitz_result = u.get_construction("markowitz")
print("Name:", markowitz_result.name)
print("Method:", markowitz_result.method)
print("Selected assets:", markowitz_result.selected_assets)
print("Params:", markowitz_result.params)
print("Metrics:", markowitz_result.metrics_insample)
markowitz_result.weights

Name: markowitz
Method: markowitz_max_sharpe
Selected assets: ['AAPL', 'AMZN', 'GOOG']
Params: {'ret_kind': 'simple', 'allow_short': False, 'ann_factor': 252, 'construction_start': '2020-01-01', 'construction_end': '2020-06-30'}
Metrics: {'n_selected': 3, 'expected_return': 0.6136055244463806, 'volatility': 0.31271339562744177, 'sharpe_m': 1.9621977600774532, 'meta_n_assets': 4, 'meta_success': True, 'meta_message': 'Optimization terminated successfully', 'meta_rf_per_period': 0.0, 'meta_objective': 0.12360684038792677, 'meta_allow_short': False, 'meta_ret_kind_used': 'simple'}


AAPL    4.800076e-01
AMZN    5.199924e-01
GOOG    1.326521e-16
MSFT    0.000000e+00
dtype: float64

## 4. Compare in-sample metrics across constructions

In [7]:
u.compare_insample_metrics()

,method,n_selected,expected_return,volatility,sharpe_m,meta_n_assets,meta_success,meta_message,meta_rf_per_period,meta_objective,...,meta_ret_kind_used,meta_nrp_method,meta_nrp_min_vol,meta_nrp_sigma,meta_hrp_n_clusters,meta_hrp_distance,meta_hrp_clustering,meta_hrp_clusters,meta_hrp_inner_n_assets,meta_hrp_outer_n_assets
name,,,,,,,,,,,,,,,,,,,,,
equal_weight,equal_weight,4,0.472394,0.279167,1.692155,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hrp_style,HRP-style,4,0.519825,0.287039,1.810992,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3.0,corr_distance,hierarchical_clusters,"[[MSFT, GOOG], [AAPL], [AMZN]]",1.0,3.0
markowitz,markowitz_max_sharpe,3,0.613606,0.312713,1.962198,4.0,True,Optimization terminated successfully,0.0,0.123607,...,simple,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
naive_risk_parity,Naive Risk Parity (1/sigma),4,0.460697,0.277399,1.660775,NaN,NaN,NaN,NaN,NaN,...,NaN,inverse_vol,1.000000e-08,"{'AAPL': 0.023131691550599295, 'MSFT': 0.01900...",NaN,NaN,NaN,NaN,NaN,NaN


## 5. Fixed-weight backtest

The weights are built on the first half of 2020, then evaluated on the second half of 2020.

In [8]:
bt = Backtester(u, "markowitz", ann_factor=252)
bt_result = bt.run_and_attach(
    start_date="2020-07-01",
    end_date="2020-12-31",
    notes="Fixed-weight out-of-sample backtest.",
)

bt_result.summary_metrics

{'n_periods': 132,
 'total_return': 0.3247601414658765,
 'annualized_return': 0.7106892217456247,
 'annualized_volatility': 0.3117303603593294,
 'sharpe_ratio': 2.2798203579735326,
 'max_drawdown': -0.1826995330623956}

In [9]:
bt_result.cumulative_returns.tail()

Date
2020-12-25    0.301634
2020-12-28    0.341359
2020-12-29    0.358005
2020-12-30    0.343052
2020-12-31    0.324760
Freq: B, Name: markowitz, dtype: float64

## 6. Monte Carlo on the same stored construction

In [10]:
mc = MonteCarloEngine(u, "markowitz", seed=42)
mc_result = mc.run_and_attach(
    horizon=30,
    n_simulations=500,
    initial_value=1.0,
    notes="Simple multivariate normal Monte Carlo.",
)

mc_result.summary_metrics

{'mean_terminal_value': 1.0746636883047571,
 'median_terminal_value': 1.0718802159329113,
 'std_terminal_value': 0.11567032193639057,
 'min_terminal_value': 0.7536595673171775,
 'max_terminal_value': 1.4452717385259932,
 'prob_loss': 0.278,
 'mean_terminal_return': 0.07466368830475723}

In [11]:
mc_result.simulated_paths.iloc[:5, :5]

,sim_0,sim_1,sim_2,sim_3,sim_4
step,,,,,
0,1.000000,1.000000,1.000000,1.000000,1.000000
1,0.997532,1.042647,1.002150,0.994272,0.994550
2,1.008852,1.068055,0.987774,0.976946,0.997397
3,1.004070,1.045230,1.000222,0.960004,0.969805
4,0.998814,1.108093,1.013322,0.951621,0.953446


## 7. Results are attached back into the stored construction

In [12]:
stored = u.get_construction("markowitz")
print(type(stored.backtest_result).__name__)
print(type(stored.mc_result).__name__)
print(stored.backtest_result.summary_metrics)
print(stored.mc_result.summary_metrics)

BacktestResult
MonteCarloResult
{'n_periods': 132, 'total_return': 0.3247601414658765, 'annualized_return': 0.7106892217456247, 'annualized_volatility': 0.3117303603593294, 'sharpe_ratio': 2.2798203579735326, 'max_drawdown': -0.1826995330623956}
{'mean_terminal_value': 1.0746636883047571, 'median_terminal_value': 1.0718802159329113, 'std_terminal_value': 0.11567032193639057, 'min_terminal_value': 0.7536595673171775, 'max_terminal_value': 1.4452717385259932, 'prob_loss': 0.278, 'mean_terminal_return': 0.07466368830475723}


## 8. Evaluate all constructions at once

In [13]:
all_bt = Backtester.run_all(
    u,
    start_date="2020-07-01",
    end_date="2020-12-31",
    ann_factor=252,
    attach=True,
)

{name: result.summary_metrics for name, result in all_bt.items()}

{'equal_weight': {'n_periods': 132,
  'total_return': 0.2501002343812391,
  'annualized_return': 0.5313572050716688,
  'annualized_volatility': 0.25583799600161533,
  'sharpe_ratio': 2.0769284210165324,
  'max_drawdown': -0.16550208519812304},
 'markowitz': {'n_periods': 132,
  'total_return': 0.3247601414658765,
  'annualized_return': 0.7106892217456247,
  'annualized_volatility': 0.3117303603593294,
  'sharpe_ratio': 2.2798203579735326,
  'max_drawdown': -0.1826995330623956},
 'naive_risk_parity': {'n_periods': 132,
  'total_return': 0.24109505462423053,
  'annualized_return': 0.5103665501529422,
  'annualized_volatility': 0.25299344766285464,
  'sharpe_ratio': 2.017311336983989,
  'max_drawdown': -0.16379358358056628},
 'hrp_style': {'n_periods': 132,
  'total_return': 0.2772587492409535,
  'annualized_return': 0.5954972340586258,
  'annualized_volatility': 0.2709406004045916,
  'sharpe_ratio': 2.1978885156723593,
  'max_drawdown': -0.17147300522455056}}

In [14]:
all_mc = MonteCarloEngine.run_all(
    u,
    horizon=20,
    n_simulations=250,
    initial_value=1.0,
    attach=True,
    seed=123,
)

{name: result.summary_metrics for name, result in all_mc.items()}

{'equal_weight': {'mean_terminal_value': 1.035914387519515,
  'median_terminal_value': 1.0343604785998746,
  'std_terminal_value': 0.08319647450471436,
  'min_terminal_value': 0.8088897540922386,
  'max_terminal_value': 1.3457469261820623,
  'prob_loss': 0.348,
  'mean_terminal_return': 0.03591438751951488},
 'markowitz': {'mean_terminal_value': 1.0389973485441946,
  'median_terminal_value': 1.0393113871909831,
  'std_terminal_value': 0.09316035525858672,
  'min_terminal_value': 0.8082209692727722,
  'max_terminal_value': 1.3873778494293467,
  'prob_loss': 0.332,
  'mean_terminal_return': 0.03899734854419485},
 'naive_risk_parity': {'mean_terminal_value': 1.0378795345510816,
  'median_terminal_value': 1.0321213469897483,
  'std_terminal_value': 0.0846634619329222,
  'min_terminal_value': 0.783910957887964,
  'max_terminal_value': 1.2857718240683271,
  'prob_loss': 0.336,
  'mean_terminal_return': 0.03787953455108157},
 'hrp_style': {'mean_terminal_value': 1.038225652702368,
  'median_t

## 9. Per-construction output paths

These are the folder helpers available on the universe for organizing exports.

In [15]:
print(u.get_construction_dir("markowitz"))
print(u.get_backtest_dir("markowitz"))
print(u.get_mc_dir("markowitz"))
print(u.get_plot_dir("weights", "markowitz"))

c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\constructions\markowitz
c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\backtests\markowitz
c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\monte_carlo\markowitz
c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\plots\weights\markowitz
